# Laboratorio 16 — Simulación de migración

**Duración: 60 minutos.**

Este laboratorio usa casi todo lo anterior. Hoy no aprendes una
función nueva: haces el trabajo completo.

Una tabla vive en una base relacional. Hay que llevarla a Iceberg **sin apagar el
origen**, porque el origen sigue recibiendo movimientos mientras dura la migración. Eso
es exactamente lo que le va a tocar a la DGT.

---

### Las cinco etapas

Toda migración, sea de una tabla o de cientos, tiene las mismas cinco:

1. **Mirar el origen.** Qué hay, cuántas filas, y cómo se sabe qué cambió.
2. **Carga inicial.** Una foto completa, y anotar hasta dónde llegaste.
3. **Detectar cambios.** Qué se movió en el origen desde esa foto.
4. **Carga incremental.** Meter solo eso, sin duplicar lo que ya estaba.
5. **Validar.** Comprobar que los dos lados dicen lo mismo.

Hoy lo haces con **una** tabla y **dos** cargas. En el SII son cientos de tablas y todas
las noches. Lo que no cambia es el procedimiento.


## Paso 0 — Qué es migrar, y qué es la marca de agua

Antes de tocar nada, una idea que es la que sostiene todo el laboratorio.

**Migrar no es copiar una vez.** Si copias la tabla entera y te vas, para cuando termines
el origen ya cambió. Y volver a copiarla entera todas las noches no escala: en el SII son
millones de filas.

Lo que se hace es copiar una vez y después **solo lo que se movió**. Para eso hace falta
saber *hasta dónde copiaste*, y eso es la **marca de agua**: la fecha de actualización más
nueva que alcanzaste a traer. Todo lo que en el origen tenga una fecha posterior a esa
marca es lo que falta.

En este laboratorio la columna se llama `actualizado_en` y la mantiene el sistema de
origen. En una migración real, si esa columna no existe, lo primero que hay que negociar
con el dueño del origen es que exista.

## Paso 1 — Mirar el origen

Primero, a tu espacio de trabajo.

In [1]:
%%sql
-- Celda 1.1
USE mi_espacio

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/25 18:11:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ahora abrimos el origen. Es una base PostgreSQL que vive al lado, y Spark la lee por
JDBC sin copiar nada todavía: `origen_pg` es una **vista**, una ventana a la tabla de
allá.

Fíjate en que esto es SQL corriente. No hay Python, no hay conector especial de Iceberg.
Es la forma estándar de Spark de mirar una base relacional.

In [2]:
%%sql
-- Celda 1.2
CREATE TEMPORARY VIEW origen_pg USING jdbc OPTIONS (
    url      'jdbc:postgresql://iceberg-postgres:5432/origen',
    dbtable  'contribuyentes',
    user     'lector',
    password ''
)

Veamos qué hay del otro lado. Veinte contribuyentes, cada uno con su fecha de
actualización.

In [3]:
%%sql
-- Celda 1.3
SELECT * FROM origen_pg ORDER BY rut

rut,razon_social,segmento,actualizado_en
76000262-3,Boldo Importadora EIRL,MICRO,2026-09-01 08:00:00
76000503-7,Calafate Ferreteria EIRL,PEQUENA,2026-09-01 08:05:00
76002248-9,Andes Consultores SpA,MEDIANA,2026-09-01 08:10:00
76002652-2,Calafate Constructora S.A.,GRANDE,2026-09-01 08:15:00
76002830-4,Huemul Alimentos Ltda.,MICRO,2026-09-01 08:20:00
76004358-3,Ulmo Constructora Ltda.,PEQUENA,2026-09-01 08:25:00
76005022-9,Andes Distribuidora EIRL,MEDIANA,2026-09-01 08:30:00
76005135-7,Huemul Servicios SpA,GRANDE,2026-09-01 08:35:00
76005575-1,Copihue Consultores EIRL,MICRO,2026-09-01 08:40:00
76005701-0,Maiten Constructora Ltda.,PEQUENA,2026-09-01 08:45:00


## Paso 2 — Carga inicial

Partimos de cero, por si ya habías corrido este laboratorio.

In [4]:
%%sql
-- Celda 2.1
DROP TABLE IF EXISTS contribuyentes_lab16

La carga inicial: la foto completa. Un `CREATE TABLE ... AS SELECT` que lee la vista
JDBC y deja una tabla Iceberg.

Esta es la etapa 2 de las cinco, y es la única que copia todo.

In [5]:
%%sql
-- Celda 2.2
CREATE TABLE contribuyentes_lab16 USING iceberg AS
SELECT * FROM origen_pg

¿Cuántos quedaron de este lado?

In [6]:
%%sql
-- Celda 2.3
SELECT count(*) AS contribuyentes FROM contribuyentes_lab16

contribuyentes
20


Veinte, los mismos. Ahora **anota la marca de agua**: la fecha de actualización más
nueva que alcanzaste a traer. La vas a escribir con tus manos en el paso 4, así que
tenla a la vista.

In [7]:
%%sql
-- Celda 2.4
SELECT max(actualizado_en) AS marca_de_agua FROM contribuyentes_lab16

marca_de_agua
2026-09-01 09:35:00


## Paso 3 — El origen sigue vivo

Aquí no ejecutas nada en el cuaderno.

Mientras hacías la carga inicial, el sistema de la DGT siguió trabajando: entraron
contribuyentes nuevos y a otros los reclasificaron. Para simularlo, abre una terminal en
la carpeta del repositorio y corre el proceso que mueve el origen:

```bash
bin/mover-origen-lab16.sh
```

Cambia el segmento de tres contribuyentes y agrega dos nuevos, todos con una fecha de
actualización posterior a tu marca de agua.

En el cuaderno no vas a ver nada. Tu tabla Iceberg sigue con sus veinte filas, y así tiene
que ser: **una migración no se entera de los cambios, los va a buscar**. Eso es el paso 4.

> Si repites este laboratorio, antes deja el origen como estaba con
> `bin/mover-origen-lab16.sh --reset`.

## Paso 4 — Detectar qué cambió

Ahora le preguntas al origen qué se movió desde tu marca de agua. Una sola condición,
y es toda la detección de cambios.

In [8]:
%%sql
-- Celda 4.1
SELECT * FROM origen_pg
WHERE actualizado_en > TIMESTAMP '2026-09-01 09:35:00'
ORDER BY rut

rut,razon_social,segmento,actualizado_en
76000262-3,Boldo Importadora EIRL,PEQUENA,2026-09-15 12:00:00
76002248-9,Andes Consultores SpA,GRANDE,2026-09-15 12:00:00
76005135-7,Huemul Servicios SpA,MEDIANA,2026-09-15 12:00:00
77129445-1,Lenga Transportes SpA,MICRO,2026-09-15 12:00:00
78440174-7,Copihue Alimentos EIRL,PEQUENA,2026-09-15 12:00:00


## Paso 5 — Carga incremental

**Cinco filas.** Tres que ya tenías y cambiaron de segmento, y dos que no existían.

Ahora las metes con el `MERGE` del laboratorio 03. La llave es el RUT: si el contribuyente
ya está, se actualiza; si no está, se inserta. Una sola sentencia hace las dos cosas, y no
hay que saber de antemano cuál es cuál.

In [9]:
%%sql
-- Celda 5.1
MERGE INTO contribuyentes_lab16 AS destino
USING (
    SELECT * FROM origen_pg
    WHERE actualizado_en > TIMESTAMP '2026-09-01 09:35:00'
) AS origen
ON destino.rut = origen.rut
WHEN MATCHED THEN UPDATE SET
    destino.razon_social   = origen.razon_social,
    destino.segmento       = origen.segmento,
    destino.actualizado_en = origen.actualizado_en
WHEN NOT MATCHED THEN INSERT *

¿Y ahora cuántos hay?

In [10]:
%%sql
-- Celda 5.2
SELECT count(*) AS contribuyentes FROM contribuyentes_lab16

contribuyentes
22


## Paso 6 — Validar

**Veintidós.** Veinte más los dos nuevos. Los tres que cambiaron no sumaron filas,
porque se actualizaron.

Pero contar no basta. Dos tablas pueden tener el mismo número de filas y decir cosas
distintas. Comparamos los dos lados con una **suma de control**, que es un número que
cambia si cambia cualquier dato.

In [11]:
%%sql
-- Celda 6.1
SELECT
    (SELECT count(*) FROM origen_pg)                                        AS filas_origen,
    (SELECT count(*) FROM contribuyentes_lab16)                                          AS filas_iceberg,
    (SELECT sum(crc32(concat(rut, razon_social, segmento))) FROM origen_pg) AS control_origen,
    (SELECT sum(crc32(concat(rut, razon_social, segmento))) FROM contribuyentes_lab16)   AS control_iceberg

filas_origen,filas_iceberg,control_origen,control_iceberg
22,22,39995702816,39995702816


## Paso 7 — Lo que quedó escrito

**Los cuatro números cuadran.** La migración está validada, y no porque alguien diga
que salió bien: porque los dos lados dan lo mismo.

Última mirada, a la libreta.

In [12]:
%%sql
-- Celda 7.1
SELECT snapshot_id, committed_at, operation
FROM mi_espacio.contribuyentes_lab16.snapshots
ORDER BY committed_at

snapshot_id,committed_at,operation
2550665139341160300,2026-09-25 18:11:43.816,append
7138320268844862618,2026-09-25 18:11:52.668,overwrite


## Cierre

**Dos páginas.** La primera es la carga inicial, la segunda la incremental. Si mañana
alguien pregunta qué trajo la carga del martes, está ahí, y con el `VERSION AS OF` del
laboratorio 01 se puede ver la tabla como quedó después de cada una.

Las cinco etapas, ahora con lo que hiciste en cada una:

| Etapa | Lo que hiciste |
|---|---|
| Mirar el origen | la vista JDBC y un `SELECT` |
| Carga inicial | `CREATE TABLE ... AS SELECT`, y anotar la marca de agua |
| Detectar cambios | `WHERE actualizado_en > marca` |
| Carga incremental | `MERGE`, que actualiza e inserta en una sentencia |
| Validar | conteo y suma de control, los dos lados |

**Qué cambia entre una tabla y cientos.** El procedimiento, nada. Lo que cambia es que
alguien tiene que guardar la marca de agua de cada tabla en alguna parte, y que las cargas
se encadenan y se vigilan. Eso es orquestación, y es una decisión de plataforma.

**Qué no cambia nunca:** que el origen sigue vivo mientras migras, y que por eso la carga
inicial no alcanza.
